# 2.4 Wedding Planner
In this lab you will build a multi-agent wedding planner.

> Note: This lab has been updated since filming to make the agent performance more robust to errors and to limit run time. In particular: 
> 1) added error handling for MCP failures, 
> 2) limited the number of searches. 
> 
> It's worth noting that, where possible, tool errors should be returned to the agent rather than failing so that the agent can adjust its invocation and try again.

In [1]:
from dotenv import load_dotenv

load_dotenv()

True

## Import Libraries and Singletons

In [12]:
import asyncio
import os
from pprint import pprint
from typing import Any, Dict  # noqa: UP035

from mcp import ClientSession, StdioServerParameters
from mcp.client.stdio import stdio_client
from langchain_mcp_adapters.tools import load_mcp_tools
from langchain.agents import AgentState, create_agent
from langchain.messages import HumanMessage, ToolMessage
from langchain.tools import ToolRuntime, tool
from langchain_community.utilities import SQLDatabase
from langchain_mcp_adapters.client import MultiServerMCPClient
from langgraph.types import Command
from mcp.shared.exceptions import McpError
from mcp.types import CallToolResult, TextContent
from tavily import TavilyClient

tavily_client = TavilyClient()
db = SQLDatabase.from_uri("sqlite:///resources/Chinook.db")
RETRYABLE_MCP_CODES = {-32603}

sandbox = os.path.abspath("sandbox")
os.makedirs(sandbox, exist_ok=True)

## Setup Tools
### Travel Server by Kiwi

In [3]:
class RetryMCPInterceptor:
    """Intercept MCP tool calls: retry transient failures, surface all errors gracefully.

    - Retryable McpError codes (e.g. -32603): retry with exponential backoff.
    - Non-retryable McpError codes (e.g. -32602): return error message immediately.
    - Any other exception (fetch failed, network errors, etc.): retry then return error message.
    """

    def __init__(self, max_retries: int = 3):
        self.max_retries = max_retries

    async def __call__(self, request, handler):
        last_error = None
        for attempt in range(self.max_retries):
            try:
                return await handler(request)
            except McpError as exc:
                last_error = exc
                print(f"[MCP interceptor] {type(exc).__name__} on {request.name} "
                      f"(code {exc.error.code}, attempt {attempt+1}/{self.max_retries}): {exc}")
                if exc.error.code not in RETRYABLE_MCP_CODES:
                    return CallToolResult(
                        content=[TextContent(type="text", text=f"Tool call failed (non-retryable): {exc}")],
                        isError=False,
                    )
            except Exception as exc:
                last_error = exc
                print(f"[MCP interceptor] {type(exc).__name__} on {request.name} "
                      f"(attempt {attempt+1}/{self.max_retries}): {exc}")

            if attempt < self.max_retries - 1:
                await asyncio.sleep(2 ** attempt)

        print(f"[MCP interceptor] all {self.max_retries} retries exhausted for {request.name}")
        return CallToolResult(
            content=[TextContent(type="text", text=f"Tool call failed after {self.max_retries} attempts: {last_error}")],
            isError=False,
        )

client = MultiServerMCPClient(
    {
        "travel_server": {
                "transport": "streamable_http",
                "url": "https://mcp.kiwi.com"
            }
    },
    tool_interceptors=[RetryMCPInterceptor()],
)

tools = await client.get_tools()

### Web Search tool by Tavily

In [4]:
@tool
def web_search(query: str, search_number: int, max_search_number: int) -> Dict[str, Any]:
    """Search the web for information. You must track your search count by providing
    search_number (starting at 1) and max_search_number on every call.
    Queries must use only plain text characters. Do not use accented or special characters
      (e.g., use 'capacite' instead of 'capacité').
    """
    if search_number > max_search_number:
        return {"message": "Search limit reached. Please summarize your findings and provide your final answer."}
    try:
        return tavily_client.search(query)
    except Exception as e:
        return {"error": str(e)}

### Query SQL Database

In [5]:
@tool
def query_playlist_db(query: str) -> str:
    """Query the database for playlist information"""

    try:
        return db.run(query)
    except Exception as e:
        return f"Error querying database: {e}"

@tool
def get_table_info(query: str) -> str:
    """Obtain information from a specific table on the database"""

    try:
        return db.table_info
    except Exception as e:
        return f"Error: {e}"

### MCP file system

In [13]:
server_params = StdioServerParameters(
    command="npx",
    args=[
        "-y",
        "@modelcontextprotocol/server-filesystem",
        sandbox,
    ],
)

## Create and Update State

In [6]:
class WeddingState(AgentState):
    origin: str
    destination: str
    guest_count: str
    genre: str

@tool
def update_state(origin: str, destination: str, guest_count: str, genre: str, runtime: ToolRuntime) -> str:
    """Update the state when you know all of the values: origin, destination, guest_count, genre.
    This tool must be called alone, without any other tool calls. It must complete and return to make,
    the information available to other tools."""
    return Command(update={
        "origin": origin,
        "destination": destination,
        "guest_count": guest_count,
        "genre": genre,
        "messages": [ToolMessage("Successfully updated state", tool_call_id=runtime.tool_call_id)]}
        )

## Create Subagents
### Travel agent

In [7]:
travel_agent = create_agent(
    model="ollama:qwen3.5:4b",
    tools=tools,
    system_prompt="""
    You are a travel agent. Search for flights to the desired destination wedding location.
    You are not allowed to ask any more follow up questions, you must find the best flight options based on the following criteria:
    - Price (lowest, economy class)
    - Duration (shortest)
    - Date (time of year which you believe is best for a wedding at this location)
    To make things easy, only look for one ticket, one way.
    You may need to make multiple searches to iteratively find the best options.
    You will be given no extra information, only the origin and destination. It is your job to think critically about the best options.
    If the MCP tool fails, returns malformed output, or does not give you usable flight results, try the tool again.
    Once you have found the best options, let the user know your shortlist of options.
    """
)

@tool
async def search_flights(runtime: ToolRuntime) -> str:
    """Travel agent searches for flights to the desired destination wedding location."""
    origin = runtime.state["origin"]
    destination = runtime.state["destination"]
    response = await travel_agent.ainvoke({"messages": [HumanMessage(content=f"Find flights from {origin} to {destination}")]})
    return response['messages'][-1].content

### Venue agent

In [8]:
venue_agent = create_agent(
    model="ollama:qwen3.5:4b",
    tools=[web_search],
    system_prompt="""
    You are a venue specialist. Search for venues in the desired location, and with the desired capacity.
    You are not allowed to ask any more follow up questions, you must find the best venue options based on the following criteria:
    - Price (lowest)
    - Capacity (exact match)
    - Reviews (highest)
    You may need to make multiple searches to iteratively find the best options.
    You have a suggested limit of 12 web searches. Count every web_search call you make.
    After 12 searches, you should stop searching and summarize the best options you have
    found so far.
    """
)

@tool
def search_venues(runtime: ToolRuntime) -> str:
    """Venue agent chooses the best venue for the given location and capacity."""
    destination = runtime.state["destination"]
    capacity = runtime.state["guest_count"]
    query = f"Find wedding venues in {destination} for {capacity} guests"
    response = venue_agent.invoke({"messages": [HumanMessage(content=query)]})
    return response['messages'][-1].content

### Playlist agent

In [9]:
playlist_agent = create_agent(
    model="ollama:qwen3.5:4b",
    tools=[query_playlist_db, get_table_info],
    system_prompt="""
    You are a playlist specialist. Query the sql database and curate the perfect playlist for a wedding given a genre.
    Once you have your playlist, calculate the total duration and cost of the playlist, each song has an associated price.
    If you run into errors when querying the database, try to fix them by making changes to the query.
    Do not come back empty handed, keep trying to query the db until you find a list of songs.

    This is a SQLite database. Before writing any data queries, first discover the schema and the table relationships
    """
)

@tool
def suggest_playlist(runtime: ToolRuntime) -> str:
    """Playlist agent curates the perfect playlist for the given genre."""
    genre = runtime.state["genre"]
    query = f"Find {genre} tracks for wedding playlist"
    response = playlist_agent.invoke({"messages": [HumanMessage(content=query)]})
    return response['messages'][-1].content

## Main Coordinator

In [19]:
async with stdio_client(server_params) as (read, write):
    async with ClientSession(read, write) as session:
        await session.initialize()

        filesystem_tools = await load_mcp_tools(session)
        tools = [search_flights, search_venues, suggest_playlist, update_state] + filesystem_tools

        coordinator = create_agent(
            model="ollama:qwen3.5:4b",
            tools=tools,
            state_schema=WeddingState,
            system_prompt="""
            You are a wedding coordinator.
            First find all the information you need to update the state. When you have the information, update the state.
            Once that has completed and returned, you can delegate the tasks
            to your specialists for flights, venues, and playlists.
            Once you have received their answers, coordinate the perfect wedding for me.
            Write a markdown `wedding_plan.md`, with your conclusions.
            """
        )

        response = await coordinator.ainvoke(
            {
                "messages": [
                    HumanMessage(content="I'm from Edinburgh, and I'd like a wedding in Paris in 60 days for 100 guests, with a playlist starting with jazz and ending with rock and roll.")],
            },
            config={"tags": ["WP"], "recursion_limit": 50},
            #tag traces to make them easy to find in Langsmith. Increase number of steps the agent can take to 40.
        )

## Check the results

In [21]:
for message in response["messages"]:
    if message.content:
        pprint(message.content)
        print("-"*60)

("I'm from Edinburgh, and I'd like a wedding in Paris in 60 days for 100 "
 'guests, with a playlist starting with jazz and ending with rock and roll.')
------------------------------------------------------------
'Successfully updated state'
------------------------------------------------------------
("I've searched for flights from Edinburgh to Paris in June and early July "
 '2027. Here are my top options based on price, duration, and timing:\n'
 '\n'
 '## **Best Options Found**\n'
 '\n'
 '### **1. Cheapest Option - Best Value**\n'
 '- **Date:** June 15, 2027 (Wednesday)\n'
 '- **Price:** €47 EUR\n'
 '- **Route:** Edinburgh → Paris Beauvais-City (BVA)\n'
 '- **Airline:** Ryanair\n'
 '- **Duration:** 1 hour 20 minutes (direct flight)\n'
 '- **Departure:** 14:05, Arrival: 16:45\n'
 '- **Cabin:** Economy\n'
 '\n'
 '### **2. Mid-Range Option**\n'
 '- **Date:** June 20, 2027 (Saturday)\n'
 '- **Price:** €75 EUR\n'
 '- **Route:** Edinburgh → Paris Beauvais-City (BVA)\n'
 '- **Airline:** 